In [ ]:
# Future Machine Learning Experiments for ClaimVision AI

This notebook captures future ML research for the ClaimVision AI damage claim verification system. It includes templates for fraud detection, severity prediction, and claim approval prediction using Random Forest, XGBoost, LightGBM, and Logistic Regression.
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Objective

The objective is to prototype three future model tracks for the claims verification pipeline:
- **Fraud Detection Model:** identify suspicious or fraudulent claims based on claim metadata and evidence signals.
- **Severity Prediction Model:** estimate the likely damage severity category from claim features and image-derived signals.
- **Claim Approval Prediction Model:** predict whether a claim should be approved, rejected, or routed for manual review.

This notebook is intentionally exploratory and designed for future enhancements, not production deployment.
</VSCode.Cell>
<VSCode.Cell language="python">
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

# Optional imports for XGBoost / LightGBM if installed
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

RANDOM_STATE = 42

# Placeholder dataset for demonstration; replace with actual feature engineering on real claim data.
example_rows = [
    {
        "claim_id": f"C-{i:03d}",
        "policy_holder_age": np.random.randint(18, 75),
        "claim_amount": np.round(np.random.uniform(500, 15000), 2),
        "num_images": np.random.randint(1, 6),
        "valid_image": np.random.choice([0, 1], p=[0.3, 0.7]),
        "damage_visible": np.random.choice([0, 1], p=[0.4, 0.6]),
        "claim_source": np.random.choice(["mobile_app", "email", "agent", "portal"]),
        "issue_type": np.random.choice(["water", "fire", "theft", "collision"]),
        "history_flag": np.random.choice([0, 1], p=[0.6, 0.4]),
        "fraud_label": np.random.choice([0, 1], p=[0.85, 0.15]),
        "severity_label": np.random.choice(["low", "medium", "high"], p=[0.45, 0.35, 0.2]),
        "approval_label": np.random.choice(["approved", "rejected", "manual_review"], p=[0.55, 0.25, 0.2]),
    }
    for i in range(1, 201)
]

df = pd.DataFrame(example_rows)
df.head()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Features

The example feature set includes:
- claim metadata: `policy_holder_age`, `claim_amount`, `num_images`
- image quality proxies: `valid_image`, `damage_visible`
- categorical context: `claim_source`, `issue_type`
- claimant behavior signal: `history_flag`

In real experiments, expand this to include:
- image embeddings or object detection outputs
- text-derived features from claim descriptions
- temporal signals such as claim age or submission time
- external fraud indicators and policy metadata
</VSCode.Cell>
<VSCode.Cell language="python">
# Feature engineering helpers
categorical_features = ["claim_source", "issue_type"]
label_encoders = {feature: LabelEncoder().fit(df[feature]) for feature in categorical_features}
for feature, encoder in label_encoders.items():
    df[f"{feature}_encoded"] = encoder.transform(df[feature])

numeric_features = ["policy_holder_age", "claim_amount", "num_images", "valid_image", "damage_visible", "history_flag"]
encoded_features = [f"{feature}_encoded" for feature in categorical_features]
feature_columns = numeric_features + encoded_features

scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

print("Feature columns:", feature_columns)
print(df[feature_columns].head())
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Training Pipeline

The training pipeline in this notebook includes:
1. Train/test split with a reproducible random seed.
2. Label encoding for categorical features.
3. Scaling numeric features for models that benefit from normalization.
4. Model training using Random Forest, Logistic Regression, XGBoost, and LightGBM.
5. Evaluation using accuracy, precision, recall, F1-score, ROC AUC, and classification reports.
</VSCode.Cell>
<VSCode.Cell language="python">
# Generic training and evaluation helper

def train_and_evaluate(model, X_train, X_test, y_train, y_test, positive_label=None):
    model_name = type(model).__name__
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    scores = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }
    if positive_label is not None and hasattr(model, "predict_proba"):
        try:
            y_prob = model.predict_proba(X_test)
            if isinstance(y_prob, np.ndarray) and y_prob.shape[1] > 1:
                scores["roc_auc"] = roc_auc_score(y_test, y_prob, multi_class="ovo", average="weighted")
            else:
                scores["roc_auc"] = roc_auc_score(y_test, y_prob[:, 1], pos_label=positive_label)
        except Exception:
            scores["roc_auc"] = None
    else:
        scores["roc_auc"] = None

    print(f"=== {model_name} Results ===")
    print(pd.Series(scores))
    print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))
    return scores
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Experiment 1: Fraud Detection Model

Use a binary classification model to predict whether a claim is likely fraudulent.
This is a key signal for risk-based claim triage and early investigation.
</VSCode.Cell>
<VSCode.Cell language="python">
fraud_df = df.copy()
X = fraud_df[feature_columns]
y = fraud_df["fraud_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

fraud_models = [
    RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100),
    LogisticRegression(random_state=RANDOM_STATE, max_iter=500),
]
if XGBClassifier is not None:
    fraud_models.append(XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=RANDOM_STATE))
if LGBMClassifier is not None:
    fraud_models.append(LGBMClassifier(random_state=RANDOM_STATE))

fraud_results = []
for model in fraud_models:
    fraud_results.append(train_and_evaluate(model, X_train, X_test, y_train, y_test, positive_label=1))

pd.DataFrame(fraud_results)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Experiment 2: Severity Prediction Model

Model severity as a multiclass target using claim metadata and evidence signals.
This can help prioritize claims and route them to the right claims adjuster.
</VSCode.Cell>
<VSCode.Cell language="python">
severity_df = df.copy()
X = severity_df[feature_columns]
y = severity_df["severity_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

severity_models = [
    RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100),
    LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, multi_class="multinomial"),
]
if XGBClassifier is not None:
    severity_models.append(XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=RANDOM_STATE))
if LGBMClassifier is not None:
    severity_models.append(LGBMClassifier(random_state=RANDOM_STATE))

severity_results = []
for model in severity_models:
    severity_results.append(train_and_evaluate(model, X_train, X_test, y_train, y_test))

pd.DataFrame(severity_results)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Experiment 3: Claim Approval Prediction Model

Predict the claim approval outcome as approved, rejected, or manual review.
This supports automated decisioning and reduces workload for claim processing teams.
</VSCode.Cell>
<VSCode.Cell language="python">
approval_df = df.copy()
X = approval_df[feature_columns]
y = approval_df["approval_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

approval_models = [
    RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100),
    LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, multi_class="multinomial"),
]
if XGBClassifier is not None:
    approval_models.append(XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=RANDOM_STATE))
if LGBMClassifier is not None:
    approval_models.append(LGBMClassifier(random_state=RANDOM_STATE))

approval_results = []
for model in approval_models:
    approval_results.append(train_and_evaluate(model, X_train, X_test, y_train, y_test))

pd.DataFrame(approval_results)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Evaluation Metrics

This notebook evaluates models using:
- **Accuracy**: overall correctness of predictions.
- **Precision**: how many predicted positives are correct.
- **Recall**: how many actual positives are found.
- **F1-score**: harmonic mean of precision and recall.
- **ROC AUC**: ranking power for binary and multiclass probability outputs.

For multiclass targets, use weighted averages to account for class imbalance.
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Future Deployment Strategy

Future deployment should consider:
- Exporting the best-performing model using `joblib` or `pickle`.
- Wrapping inference in a REST API or batch scoring pipeline.
- Adding monitoring for drift, label distribution changes, and accuracy degradation.
- Using feature stores or consistent preprocessing artifacts in production.
- Running periodic retraining with fresh claim and image evidence data.

Suggested next steps:
- Collect real labeled claim data and image extraction features.
- Add SHAP or feature importance analysis to explain model decisions.
- Validate models on deployed decision outcomes and manual review feedback.
